# Imports

In [1]:
import re
import cda2
import math

import time
from datetime import datetime, timedelta

import pandas as pd
import pyarrow
from typing import Iterator, Tuple
from pyspark.sql.functions import pandas_udf

from pyspark.sql.window import Window
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import col, row_number
from pyspark.sql.functions import explode, map_keys, col

In [2]:
api = cda2.Api()

In [3]:
# Set configuration parameters to better optimize queries.

config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

In [4]:
# Start Spark and specify number of cpus to use.

api.start_spark(n_executors=100, config=config)

https://artifacts.mitre.org/artifactory/java-libs-release added as a remote repository with the name: repo-1
https://dali.mitre.org/nexus/content/repositories/mitre-caasd-releases added as a remote repository with the name: repo-2
https://dali.mitre.org/nexus/content/repositories/external-releases added as a remote repository with the name: repo-3
Ivy Default Cache set to: /home/rchong/.ivy2/cache
The jars for the packages stored in: /home/rchong/.ivy2/jars
org.mitre.spark#spark-geo_spark3.5_2.12 added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
graphframes#graphframes added as a dependency
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-04b18746-b1ca-4b66-bedd-2ce0c054d6f0;1.0
	confs: [default]


:: loading settings :: url = jar:file:/devel/data_access/software/tdp-jupyter/poetry/cache/virtualenvs/python39-QwwvzYkJ-py3.9/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mitre.spark#spark-geo_spark3.5_2.12;0.2.0 in repo-1
	found org.mitre.spark#spark-geo-core_2.12;0.2.0 in repo-1
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found net.sf.geographiclib#GeographicLib-Java;2.0 in central
	found org.ejml#ejml-core;0.43.1 in central
	found org.ejml#ejml-ddense;0.43.1 in central
	found com.esri.geometry#esri-geometry-api;2.2.4 in central
	found com.fasterxml.jackson.core#jackson-core;2.9.6 in central
	found com.google.geometry#s2-geometry;2.0.0 in central
	found com.google.guava#guava;25.1-jre in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found org.checkerframework#checker-qual;2.0.0 in central
	found com.google.errorprone#error_prone_annotations;2.1.3 in central
	found com.google.j2objc#j2objc-annotations;1.1 in central
	found org.codehaus.mojo#animal-sniffer-annotations;1.14 in central
	found com.uber#h3;4.1.1 in central
	found org.apache.spark#spark-avro_2.12;3.5.1 in central
	found org.tuka

In [5]:
year0 = "2025"
year1 = str(int(year0) + 1)

In [6]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

# retrieve tracons

In [7]:
df_tracons = (
    api.dataframe("StaticEramAirspace",
                  **dates,
                  partition_filters=api.custom_partitions(['TRACON']),
                  metadata=True)
    .select(
        'primary_key',
        'identifier',
        'center',
        'chart_date',
        'type', F.explode('geometry.modules').alias('mod'))
    .withColumn('mod_identifier', F.col('mod.identifier'))
    .select(
        'primary_key',
        'identifier',
        'center',
        'chart_date',
        'type',
        'mod_identifier',
        F.col('mod.floor').alias('floor'),
        F.col('mod.ceiling').alias('ceiling'),
        F.explode('mod.polygon.boundary').alias('boundary'))
    )

Multiple versions found: 3.1.71, 3.1.73, 3.1.74, 3.1.76, 3.1.77, 3.1.79
                                                                                

In [8]:
df_tracons = (
    api.dataframe("StaticEramAirspace",
                  **dates,
                  partition_filters=api.custom_partitions(['TRACON']),
                  metadata=True)
    .select(
        'type',
        'center',
        F.col('identifier').alias('tracon'),
        'chart_date')
    .filter(F.col("tracon") == "PWM")
    .withColumn("grouping", F.concat("center", F.lit("_"), "tracon"))
    .orderBy("grouping")
)

Multiple versions found: 3.1.71, 3.1.73, 3.1.74, 3.1.76, 3.1.77, 3.1.79
                                                                                

In [9]:
df_tracons = (
    api.dataframe("StaticEramAirspace",
                  **dates,
                  partition_filters=api.custom_partitions(['TRACON']),
                  metadata=True)
    .select (
        "type",
        "center",
        "identifier",
        "chart_date",
#        F.col("geometry.modules"),
        F.explode('geometry.modules').alias('module')
    )
    .withColumnRenamed('identifier', 'tracon')
    .filter(F.col("tracon") == "PWM")
    .withColumn("grouping", F.concat("center", F.lit("_"), "tracon", F.lit("_"), "module.identifier"))
    .orderBy("grouping")
)

Multiple versions found: 3.1.71, 3.1.73, 3.1.74, 3.1.76, 3.1.77, 3.1.79
                                                                                

In [10]:
df_tracons.count()

28

In [11]:
#df_tracons.show(1, truncate=False)
df_tracons.show()

+------+------+------+-------------+--------------------+---------------+
|  type|center|tracon|   chart_date|              module|       grouping|
+------+------+------+-------------+--------------------+---------------+
|TRACON|   ZBW|   PWM|1740009600000|{09501-1, {[{44.5...|ZBW_PWM_09501-1|
|TRACON|   ZBW|   PWM|1735171200000|{09501-1, {[{44.5...|ZBW_PWM_09501-1|
|TRACON|   ZBW|   PWM|1764201600000|{09501-1, {[{44.5...|ZBW_PWM_09501-1|
|TRACON|   ZBW|   PWM|1759363200000|{09501-1, {[{44.5...|ZBW_PWM_09501-1|
|TRACON|   ZBW|   PWM|1754524800000|{09501-1, {[{44.5...|ZBW_PWM_09501-1|
|TRACON|   ZBW|   PWM|1749686400000|{09501-1, {[{44.5...|ZBW_PWM_09501-1|
|TRACON|   ZBW|   PWM|1744848000000|{09501-1, {[{44.5...|ZBW_PWM_09501-1|
|TRACON|   ZBW|   PWM|1735171200000|{09501-2, {[{44.5...|ZBW_PWM_09501-2|
|TRACON|   ZBW|   PWM|1764201600000|{09501-2, {[{44.5...|ZBW_PWM_09501-2|
|TRACON|   ZBW|   PWM|1759363200000|{09501-2, {[{44.5...|ZBW_PWM_09501-2|
|TRACON|   ZBW|   PWM|1754524800000|{0

## keep only the newest update

In [12]:
window = Window.partitionBy("grouping").orderBy(col("chart_date").desc())

df_tracons_newest = (df_tracons
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "chart_date")
    .orderBy("grouping")
)

In [13]:
df_tracons_newest.count()

4

In [14]:
#df_tracons_newest.show(10, truncate=False)
df_tracons_newest.show(1, truncate=False)

[Stage 22:=====================================================>(399 + 1) / 400]

+------+------+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
modules_schema = T.StructType([
    T.StructField("modules", T.ArrayType(
        T.StructType([
            T.StructField("identifier", T.DoubleType(), True),
            T.StructField("floor", T.DoubleType(), True),
            T.StructField("ceiling", T.DoubleType(), True),
            T.StructField("polygon", T.StructType([
                T.StructField("boundary", T.ArrayType(
                    T.StructType([
                        T.StructField("latitude", T.DoubleType(), True),
                        T.StructField("longitude", T.DoubleType(), True),
                        T.StructField("sequence_number", T.IntegerType(), True),
                    ]), True), True),
            ]), True),
        ]), True), True),
    ])

In [ ]:
none_string = "*"
separator_string = ":"

#@F.udf(T.StringType())
@F.udf(T.MapType(T.StringType(),T.StringType()))
def extract_boundary_from_modules(array_of_modules:T.ArrayType(modules_schema)) -> map:  
    result = {}
    
    for module in array_of_modules:
        result["floor"] = f'{module.floor:.0f}'
        result["ceiling"] = f'{module.ceiling:.0f}'

        boundary_string = ""

        module.polygon.boundary.sort(key=lambda x: x.sequence_number, reverse=False)

        for point in module.polygon.boundary:
            boundary_string += (none_string if point.latitude is None else f'{point.latitude:.6f}') + " "
            boundary_string += (none_string if point.longitude is None else f'{point.longitude:.6f}') + separator_string

        # drop the trailing separator
        if len(boundary_string) > 0:
            boundary_string = boundary_string[0:len(separator_string) * -1]
           
        result["boundary"] = boundary_string
        
    return result

In [15]:
module_schema = T.StructType([
    T.StructField("module", T.StructType([
        T.StructField("identifier", T.DoubleType(), True),
        T.StructField("floor", T.DoubleType(), True),
        T.StructField("ceiling", T.DoubleType(), True),
        T.StructField("polygon", T.StructType([
            T.StructField("boundary", T.ArrayType(
                T.StructType([
                    T.StructField("latitude", T.DoubleType(), True),
                    T.StructField("longitude", T.DoubleType(), True),
                    T.StructField("sequence_number", T.IntegerType(), True),
                ]), True), True),
        ]), True),
    ]), True),
])

In [24]:
none_string = "*"
separator_string = ":"

#@F.udf(T.StringType())
@F.udf(T.MapType(T.StringType(),T.StringType()))
def extract_boundary_from_module(module:module_schema) -> map:  
    result = {}

    result["shelf"] = f'{module.identifier}'
    result["floor"] = f'{module.floor:.0f}'
    result["ceiling"] = f'{module.ceiling:.0f}'

    boundary_string = ""

    module.polygon.boundary.sort(key=lambda x: x.sequence_number, reverse=False)

    for point in module.polygon.boundary:
        boundary_string += (none_string if point.latitude is None else f'{point.latitude:.6f}') + " "
        boundary_string += (none_string if point.longitude is None else f'{point.longitude:.6f}') + separator_string

    # drop the trailing separator
    if len(boundary_string) > 0:
        boundary_string = boundary_string[0:len(separator_string) * -1]

    result["boundary"] = boundary_string
    result["shelf"] = module.identifier

    return result

In [25]:
df_tracons_dict = (
    df_tracons_newest
    .withColumn("module_dict", extract_boundary_from_module("module"))
    .select(
        "type",
        "center",
        "tracon",
        "module_dict",
    )
    .drop("module")
    .orderBy("type", "center", "tracon")
)

In [26]:
df_tracons_dict.show(1, truncate=False)

[Stage 39:>                                                         (0 + 1) / 1]

+------+------+------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|type  |center|tracon|module_dict                                                                                                                                                                                                                                                                                                                                                 

In [27]:
# expand the dictionary to columns

# https://mungingdata.com/pyspark/dict-map-to-multiple-columns/
# https://stackoverflow.com/questions/36869134/pyspark-converting-a-column-of-type-map-to-multiple-columns-in-a-dataframe

keys = ["shelf", "ceiling", "floor", "boundary"]
key_cols = list(map(lambda f: F.col("module_dict").getItem(f).alias(str(f)), keys))
final_cols = [
        "type",
        "center",
        "tracon",
             ] + key_cols

In [28]:
df_tracons_output = (
    df_tracons_dict.select(final_cols)
)

In [29]:
df_tracons_output.show(10)

[Stage 44:>                                                         (0 + 1) / 1]

+------+------+------+-------+-------+-----+--------------------+
|  type|center|tracon|  shelf|ceiling|floor|            boundary|
+------+------+------+-------+-------+-----+--------------------+
|TRACON|   ZBW|   PWM|09501-1|   3000|    0|44.583333 -70.966...|
|TRACON|   ZBW|   PWM|09501-2|   5000| 3100|44.583333 -70.966...|
|TRACON|   ZBW|   PWM|09501-3|   6000| 5100|44.583333 -70.966...|
|TRACON|   ZBW|   PWM|09501-4|  10000| 6100|44.333333 -70.866...|
+------+------+------+-------+-------+-----+--------------------+



In [ ]:
(
    df_tracons_output
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/tracons", compression="None", mode="overwrite")
)